# CH3. Online Inference


**Synthetic Nonlinear Regression**
$$
y =
10\sin(\pi x_1x_2)
+20(x_3-0.5)^2
+10x_4
+5x_5
+\epsilon
$$

## Preamble

In [26]:
# Standard Library
import os
import sys
import time
from pathlib import Path

# Numerical Processing
import numpy as np

# SmartSim and SmartRedis
from smartsim import Experiment
from smartredis import Client

# Machine Learning
import jax
import jax.numpy as jnp
import equinox as eqx

# ONNX validation
import onnx
import onnxruntime as ort

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualisation
import DataGraph as dg
import matplotlib.pyplot as plt

In [2]:
# Remove inherited Slurm environment from the interactive parent job
for key in list(os.environ):
    if key.startswith(("SLURM_", "SBATCH_", "SRUN_")):
        os.environ.pop(key)

### Directories

In [3]:
# Directory
PROJECT_DIR = Path("/scratch/project_2015384/Hanseul")
BASE_DIR = PROJECT_DIR / "Codes" / "SmartSim"
MAIN_DIR = BASE_DIR / "OnlineInference"
SCRIPT_DIR = MAIN_DIR / "scripts"
SAVE_DIR = MAIN_DIR / "model"

for directory in [PROJECT_DIR, BASE_DIR, MAIN_DIR, SCRIPT_DIR, SAVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PRODUCER_PATH = SCRIPT_DIR / "producer.py"
CONSUMER_PATH = SCRIPT_DIR / "consumer.py"
MODEL_PATH = {
    "et": SAVE_DIR / "extra_trees_regressor.onnx",
    "vr": SAVE_DIR / "voting_regressor.onnx",
    "dnn": SAVE_DIR / "dnn_regressor.onnx",
    "dnn_eqx": SAVE_DIR / "dnn_regressor.eqx"
}

### Configuration

In [4]:
# System configuration
SEED = 42
np.random.seed(SEED)
N_JOBS = len(list(os.sched_getaffinity(0)))

# SmartRedis configuration
REDIS_PORT = 2026

In [5]:
# ONNX configuration (Backend ONNX ver: 1.21.0 -> opset: 26, IR: 13)
OPSET_VERSION = 26
IR_VERSION = 13

In [6]:
# Dataset configuration
N_SAMPLES = int(1e4)
SIGMA_NOISE = 0.1

### Metric

In [7]:
# Metric computation functions (MAE, MSE, R2)
def compute_metrics(model, X_test, y_test, title="Model Evaluation Metrics", verbose=True):
    # Compute metrics for the given model and test data
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Print the computed metrics
    summary = dg.TableMaker(
        title=title,
        columns=["Metric", "Value"],
    )
    summary.add_row("MAE", f"{mae:.4f}")
    summary.add_row("MSE", f"{mse:.4f}")
    summary.add_row("R2", f"{r2:.4f}")
    
    if verbose:
        summary.display()
    
    return (mae, mse, r2)

In [8]:
# Metric computation functions (MAE, MSE, R2)
def compute_metrics_backend(y_true, y_pred, title="Model Evaluation Metrics", verbose=True):
    # Compute metrics for the given model and test data
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Print the computed metrics
    summary = dg.TableMaker(
        title=title,
        columns=["Metric", "Value"],
    )
    summary.add_row("MAE", f"{mae:.4f}")
    summary.add_row("MSE", f"{mse:.4f}")
    summary.add_row("R2", f"{r2:.4f}")
    
    if verbose:
        summary.display()
    
    return (mae, mse, r2)

### Dataset and Model I/O

#### DNN Loade Helper

In [9]:
class Linear(eqx.Module):
    weight: jax.Array
    bias: jax.Array

    def __init__(self, key, in_features, out_features, init="glorot"):
        if isinstance(init, str):
            init = {
                "glorot": jax.nn.initializers.glorot_normal(),
                "he": jax.nn.initializers.he_normal(),
                "lecun": jax.nn.initializers.lecun_normal(),
            }.get(init.lower(), jax.nn.initializers.glorot_normal())

        self.weight = init(key, (out_features, in_features))
        self.bias = jnp.zeros(out_features)

    def __call__(self, x):
        return self.weight @ x + self.bias

In [10]:
class DNN(eqx.Module):
    layers: list
    activation: callable = eqx.field(static=True)

    def __init__(self, key, input_dim, hidden_dims, output_dim,
                 activation=jax.nn.silu, init="glorot"):

        self.layers = self._build_layers(
            layer=Linear,
            key=key,
            in_dim=input_dim,
            hidden_dims=hidden_dims,
            out_dim=output_dim,
            init=init
        )

        self.activation = activation

    @staticmethod
    def _build_layers(layer, key, in_dim, hidden_dims, out_dim, init="glorot"):
        dims = [in_dim] + hidden_dims + [out_dim]
        keys = jax.random.split(key, len(dims) - 1)

        return [
            layer(
                key=keys[i],
                in_features=dims[i],
                out_features=dims[i + 1],
                init=init
            )
            for i in range(len(dims) - 1)
        ]

    @staticmethod
    def _forward(layers, x, activation):
        for layer in layers[:-1]:
            x = activation(layer(x))

        return layers[-1](x)

    def __call__(self, x):
        return self._forward(self.layers, x, self.activation)

    def predict(self, X):
        return jax.vmap(self)(X).squeeze(-1)

In [11]:
# Recreate the Equinox model structure
dnn_template = DNN(
    input_dim=10,
    output_dim=1,
    hidden_dims=[128, 128],
    activation=jax.nn.relu,
    key=jax.random.PRNGKey(SEED),
)

#### Load Models

In [12]:
# Load ONNX models (Extra Trees, Voting Regressor, and DNN)
et_model = onnx.load(MODEL_PATH["et"])
vr_model = onnx.load(MODEL_PATH["vr"])
dnn_model = onnx.load(MODEL_PATH["dnn"])
dnn_eqx_model = eqx.tree_deserialise_leaves(MODEL_PATH["dnn_eqx"], dnn_template)

## SmartSim Orchestrator (Local)

### Orchestration Setup

In [13]:
# Initialise experiment
exp = Experiment(
    name="tutorial-online-inference-local",
    launcher="local",
)

# Define orchestrator database
db = exp.create_database(
    db_nodes=1,
    port=REDIS_PORT,
    interface="lo", # lo, ib0, enp1s0f0, enp1s0f1
    inter_op_threads=1,
    intra_op_threads=2,
)

# Set up  and start the experiment
exp.generate(db, overwrite=True)
exp.start(db, block=False)

# Open a client connection to the orchestrator database
client = Client(
    address=db.get_address()[0],
    cluster=False,
)

SmartRedis Library@18-08-20:WARNING: Environment variable SR_LOG_FILE is not set. Defaulting to stdout
SmartRedis Library@18-08-20:WARNING: Environment variable SR_LOG_LEVEL is not set. Defaulting to INFO


### Enroll the ONNX and eqx models with SmartRedis

In [14]:
# DNN.onnx
start_time = time.time()
client.set_model_from_file(
    name="dnn",
    model_file=str(MODEL_PATH["dnn"]),
    backend="ONNX",
    device="CPU",
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

# DNN Equinox model
start_time = time.time()
client.set_model(
    name="dnn_eqx",
    model=jax.vmap(dnn_eqx_model),
    backend="JAX",
    example_inputs=(jnp.zeros((1, 10), dtype=jnp.float32),),
    jax_shape_specs=("batch, ...",),
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

# Extra Trees Regressor.onnx
start_time = time.time()
client.set_model_from_file(
    name="et",
    model_file=str(MODEL_PATH["et"]),
    backend="ONNX",
    device="CPU",
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

# Voting Regressor.onnx
start_time = time.time()
client.set_model_from_file(
    name="vr",
    model_file=str(MODEL_PATH["vr"]),
    backend="ONNX",
    device="CPU",
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

Model enrollment time: 0.1081 seconds


Model enrollment time: 0.0986 seconds
Model enrollment time: 0.9545 seconds
Model enrollment time: 0.8552 seconds


### Run Producer for Dataset

In [ ]:
# Create producer
producer_settings = exp.create_run_settings(
    exe="python3",
    exe_args=[
        str(PRODUCER_PATH),
        "--seed", str(SEED),
        "--n_samples", str(N_SAMPLES),
        "--sigma_noise", str(SIGMA_NOISE),
    ]
)

producer = exp.create_model(
    name="producer",
    run_settings=producer_settings,
)

# Create consumer
consumer_settings = exp.create_run_settings(
    exe="python3",
    exe_args=[str(CONSUMER_PATH)],
)

consumer = exp.create_model(
    name="consumer",
    run_settings=consumer_settings,
)

In [16]:
# Strat the producer and consumer models
exp.generate(producer, overwrite=True)
exp.generate(consumer, overwrite=True)
exp.start(producer, consumer, block=True, summary=True)

18:08:22 rc5283 SmartSim[1540056:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-online-inference-local
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/OnlineInference/tutorial-online-inference-local
Launcher: local
Models: 2
Database Status: active

=== Models ===
producer
Executable: /ROIHU_TYKKY_u2ZGue8/miniforge/envs/env1/bin/python3
Executable Arguments: /scratch/project_2015384/Hanseul/Codes/SmartSim/OnlineInference/scripts/producer.py --seed 42 --n_samples 10000 --sigma_noise 0.1
consumer
Executable: /ROIHU_TYKKY_u2ZGue8/miniforge/envs/env1/bin/python3
Executable Arguments: /scratch/project_2015384/Hanseul/Codes/SmartSim/OnlineInference/scripts/consumer.py



18:08:26 rc5283 SmartSim[1540056:JobManager] INFO producer(1541383): SmartSimStatus.STATUS_COMPLETED
18:08:27 rc5283 SmartSim[1540056:MainThread] INFO consumer(1541477): SmartSimStatus.STATUS_COMPLETED
18:08:28 rc5283 SmartSim[1540056:JobManager] INFO consumer(1541477): SmartSimStatus.STATUS_

In [17]:
# Get the predictions from the models
dataset = client.get_dataset("Friedman1_Dataset")
results = client.get_dataset("Friedman1_Inference")

X_test = dataset.get_tensor("features")
y_test = dataset.get_tensor("targets")
y_pred_dnn = results.get_tensor("dnn_predictions").squeeze()
y_pred_dnn_eqx = results.get_tensor("dnn_eqx_predictions").squeeze()
y_et = results.get_tensor("et_predictions").squeeze()
y_vr = results.get_tensor("vr_predictions").squeeze()

In [18]:
# Compute metrics for each model
dnn_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_pred_dnn, title="DNN ONNX Model")
dnn_eqx_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_pred_dnn_eqx, title="DNN eqx Model")
et_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_et, title="Extra Trees Regressor ONNX Model")
vr_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_vr, title="Voting Regressor ONNX Model")

Metric,Value
MAE,0.2522
MSE,0.1158
R2,0.9951


Metric,Value
MAE,0.2522
MSE,0.1158
R2,0.9951


Metric,Value
MAE,0.3234
MSE,0.1812
R2,0.9924


Metric,Value
MAE,0.3142
MSE,0.1718
R2,0.9928


## SmartSim Orchestrator (SLURM)

### Orchestration Setup

In [28]:
# Initialise experiment
exp = Experiment(
    name="tutorial-online-inference-slurm",
    launcher="slurm",
)

# Define orchestrator database
db = exp.create_database(
    db_nodes=1,
    port=REDIS_PORT,
    interface="ib0", # lo, ib0, enp1s0f0, enp1s0f1
    batch=True,
    time="00:10:00",
    account="project_2015384",
    inter_op_threads=1,
    intra_op_threads=2
)

db.batch_settings.set_partition("small")
db.batch_settings.batch_args["ntasks-per-node"] = "1"
db.batch_settings.batch_args["mem"] = "4G"
db.batch_settings.set_cpus_per_task(2)

# Set up  and start the experiment
exp.generate(db, overwrite=True)
exp.start(db, block=False)

# Open a client connection to the orchestrator database
client = Client(
    address=db.get_address()[0],
    cluster=False,
)

18:16:59 rc5283 SmartSim[1540056:MainThread] INFO Orchestrator launched as a batch
18:16:59 rc5283 SmartSim[1540056:MainThread] INFO While queued, SmartSim will wait for Orchestrator to run
18:16:59 rc5283 SmartSim[1540056:MainThread] INFO CTRL+C interrupt to abort and cancel launch


### Enroll the ONNX and eqx models with SmartRedis

In [29]:
# DNN.onnx
start_time = time.time()
client.set_model_from_file(
    name="dnn",
    model_file=str(MODEL_PATH["dnn"]),
    backend="ONNX",
    device="CPU",
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

# DNN Equinox model
start_time = time.time()
client.set_model(
    name="dnn_eqx",
    model=jax.vmap(dnn_eqx_model),
    backend="JAX",
    example_inputs=(jnp.zeros((1, 10), dtype=jnp.float32),),
    jax_shape_specs=("batch, ...",),
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

# Extra Trees Regressor.onnx
start_time = time.time()
client.set_model_from_file(
    name="et",
    model_file=str(MODEL_PATH["et"]),
    backend="ONNX",
    device="CPU",
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

# Voting Regressor.onnx
start_time = time.time()
client.set_model_from_file(
    name="vr",
    model_file=str(MODEL_PATH["vr"]),
    backend="ONNX",
    device="CPU",
)
end_time = time.time()
print(f"Model enrollment time: {end_time - start_time:.4f} seconds")

Model enrollment time: 0.1812 seconds
Model enrollment time: 0.0215 seconds
Model enrollment time: 1.1150 seconds
Model enrollment time: 0.9725 seconds


### Run Producer for Dataset

In [30]:
# Create producer
producer_batch_settings = exp.create_batch_settings(
    nodes=1,
    time="00:10:00",
    account="project_2015384",
)

producer_batch_settings.set_partition("small")
producer_batch_settings.batch_args["ntasks-per-node"] = "1"
producer_batch_settings.set_cpus_per_task(1)

producer_run_settings = exp.create_run_settings(
    exe=sys.executable,
    exe_args=[
        str(PRODUCER_PATH),
        "--seed", str(SEED),
        "--n_samples", str(N_SAMPLES),
        "--sigma_noise", str(SIGMA_NOISE),
    ]
)

producer = exp.create_model(
    name="producer",
    run_settings=producer_run_settings,
    batch_settings=producer_batch_settings
)

# Create consumer
consumer_batch_settings = exp.create_batch_settings(
    nodes=1,
    time="00:10:00",
    account="project_2015384",
)
consumer_batch_settings.set_partition("small")
consumer_batch_settings.batch_args["ntasks-per-node"] = "1"
consumer_batch_settings.set_cpus_per_task(1)

consumer_run_settings = exp.create_run_settings(
    exe=sys.executable,
    exe_args=[str(CONSUMER_PATH)],
)

consumer = exp.create_model(
    name="consumer",
    run_settings=consumer_run_settings,
    batch_settings=consumer_batch_settings
)

In [31]:
# Strat the producer and consumer models
exp.generate(producer, overwrite=True)
exp.generate(consumer, overwrite=True)
exp.start(producer, block=True, summary=True)
exp.start(consumer, block=True, summary=True)

18:17:19 rc5283 SmartSim[1540056:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-online-inference-slurm
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/OnlineInference/tutorial-online-inference-slurm
Launcher: slurm
Models: 1
Database Status: active

=== Models ===
producer
Batch Command: sbatch
Batch arguments:
	nodes = 1
	time = 00:10:00
	account = project_2015384
	partition = small
	ntasks-per-node = 1
	cpus-per-task = 1
Executable: /scratch/project_2015384/Hanseul/Utilities/Python/PythonSmartSim/envs/PentagonToy-3.12-x64/bin/python
Executable Arguments: /scratch/project_2015384/Hanseul/Codes/SmartSim/OnlineInference/scripts/producer.py --seed 42 --n_samples 10000 --sigma_noise 0.1
Run Command: /usr/bin/srun



18:17:25 rc5283 SmartSim[1540056:MainThread] INFO producer(311388): SmartSimStatus.STATUS_NEW
18:17:30 rc5283 SmartSim[1540056:MainThread] INFO producer(311388): SmartSimStatus.STATUS_RUNNING
18:17:35 rc5283 SmartSim[1540056:MainThread] INFO pr

In [32]:
# Get the predictions from the models
dataset = client.get_dataset("Friedman1_Dataset")
results = client.get_dataset("Friedman1_Inference")

X_test = dataset.get_tensor("features")
y_test = dataset.get_tensor("targets")
y_pred_dnn = results.get_tensor("dnn_predictions").squeeze()
y_pred_dnn_eqx = results.get_tensor("dnn_eqx_predictions").squeeze()
y_et = results.get_tensor("et_predictions").squeeze()
y_vr = results.get_tensor("vr_predictions").squeeze()

In [33]:
# Compute metrics for each model
dnn_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_pred_dnn, title="DNN ONNX Model")
dnn_eqx_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_pred_dnn_eqx, title="DNN eqx Model")
et_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_et, title="Extra Trees Regressor ONNX Model")
vr_metrics = compute_metrics_backend(y_true=y_test, y_pred=y_vr, title="Voting Regressor ONNX Model")

Metric,Value
MAE,0.2522
MSE,0.1158
R2,0.9951


Metric,Value
MAE,0.2522
MSE,0.1158
R2,0.9951


Metric,Value
MAE,0.3234
MSE,0.1812
R2,0.9924


Metric,Value
MAE,0.3142
MSE,0.1718
R2,0.9928


In [35]:
# Summarise the experiment results
print(exp.summary(style="rounded_grid"))

# Cleanup
time.sleep(3)
print("\nExperiment completed. Cleaning up...")
exp.stop(db)
exp.stop(producer, consumer)
print("Done.")

╭────┬──────────┬───────────────┬─────────┬─────────┬─────────┬─────────────────────────────────┬──────────────╮
│    │ Name     │ Entity-Type   │ JobID   │ RunID   │ Time    │ Status                          │ Returncode   │
├────┼──────────┼───────────────┼─────────┼─────────┼─────────┼─────────────────────────────────┼──────────────┤
│ 0  │ producer │ Model         │ 311388  │ 0       │ 18.7639 │ SmartSimStatus.STATUS_COMPLETED │ 0            │
├────┼──────────┼───────────────┼─────────┼─────────┼─────────┼─────────────────────────────────┼──────────────┤
│ 1  │ consumer │ Model         │ 311389  │ 0       │ 17.7491 │ SmartSimStatus.STATUS_COMPLETED │ 0            │
╰────┴──────────┴───────────────┴─────────┴─────────┴─────────┴─────────────────────────────────┴──────────────╯

Experiment completed. Cleaning up...
18:18:29 rc5283 SmartSim[1540056:MainThread] INFO Stopping model orchestrator with job name orchestrator-DK5771G33CBQ
Done.
